# Lec 02 — Linear Regression (TF 2.x)

**가설 (Hypothesis):** $H(x) = Wx + b$

**비용 (Cost / MSE):** $\text{cost}(W, b) = \frac{1}{m} \sum_{i=1}^{m} (H(x_i) - y_i)^2$

**목표:** $\arg\min_{W, b}\ \text{cost}(W, b)$

원본 강의는 TF 1.x `Session`/`placeholder` 기반이지만, 본 실습은 두 가지 스타일로 진행:
1. **저수준** — `tf.Variable` + `tf.GradientTape` (그래디언트가 어떻게 흐르는지 직접 보기)
2. **고수준** — `tf.keras.Sequential([Dense(1)])` (실무에서 쓰는 방식)

## 0. 환경 셋업

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(0)
np.random.seed(0)

print("TF:", tf.__version__)

## 1. 데이터

강의의 가장 단순한 예: $y = x$ 에 가까운 점들.

In [ ]:
x_train = tf.constant([1.0, 2.0, 3.0])
y_train = tf.constant([1.0, 2.0, 3.0])

print("x:", x_train.numpy())
print("y:", y_train.numpy())

## 2. 저수준 — `tf.Variable` + `tf.GradientTape`

`W`, `b` 를 학습 가능한 변수로 만들고, **자동 미분**으로 그래디언트를 받아 직접 업데이트.

In [ ]:
W = tf.Variable(tf.random.normal([1]), name="W")
b = tf.Variable(tf.random.normal([1]), name="b")

print("init W:", W.numpy(), "  init b:", b.numpy())

In [ ]:
lr = 0.1
history = []

for step in range(201):
    with tf.GradientTape() as tape:
        y_pred = W * x_train + b
        cost = tf.reduce_mean((y_pred - y_train) ** 2)

    dW, db = tape.gradient(cost, [W, b])
    W.assign_sub(lr * dW)
    b.assign_sub(lr * db)

    history.append(cost.numpy())
    if step % 20 == 0:
        print(f"step={step:4d}  cost={cost.numpy():.6f}  W={W.numpy()[0]:.4f}  b={b.numpy()[0]:.4f}")

print("\nfinal:  W=", W.numpy(), "  b=", b.numpy())

### 학습된 모델로 예측

In [ ]:
for x in [5.0, 2.5, 1.5]:
    pred = (W * x + b).numpy()[0]
    print(f"H({x}) = {pred:.4f}")

### Cost 곡선과 회귀선 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history)
axes[0].set_xlabel("step")
axes[0].set_ylabel("cost")
axes[0].set_title("Cost over training")
axes[0].grid(True, alpha=0.3)

xs = np.linspace(0, 4, 50)
ys = W.numpy()[0] * xs + b.numpy()[0]
axes[1].scatter(x_train.numpy(), y_train.numpy(), color="red", label="data", zorder=3)
axes[1].plot(xs, ys, label=f"H(x) = {W.numpy()[0]:.3f}x + {b.numpy()[0]:.3f}")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
axes[1].set_title("Fitted line")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

os.makedirs("../../outputs/season1/lec02", exist_ok=True)
fig.savefig("../../outputs/season1/lec02/lowlevel_fit.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. 고수준 — `tf.keras.Sequential([Dense(1)])`

동일한 문제를 Keras로. `Dense(units=1, input_shape=(1,))` 가 곧 $W x + b$ 한 개의 선형 유닛.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(1),
])

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.1),
    loss="mse",
)
model.summary()

In [ ]:
X = x_train.numpy().reshape(-1, 1)
Y = y_train.numpy().reshape(-1, 1)

hist = model.fit(X, Y, epochs=200, verbose=0)

W_k, b_k = model.layers[0].get_weights()
print(f"keras W = {W_k[0,0]:.4f},  b = {b_k[0]:.4f}")
print(f"final loss = {hist.history['loss'][-1]:.6f}")

In [ ]:
for x in [5.0, 2.5, 1.5]:
    pred = model.predict(np.array([[x]]), verbose=0)[0, 0]
    print(f"H({x}) = {pred:.4f}")

## 4. 두 방식 비교

| | 저수준 (`GradientTape`) | 고수준 (`Keras`) |
| --- | --- | --- |
| 변수 정의 | `tf.Variable` 직접 | `Dense` 레이어가 자동 |
| 그래디언트 | `tape.gradient` | `optimizer`가 알아서 |
| 업데이트 | `assign_sub` 수동 | `model.fit` |
| 언제 쓰나 | 학습 루프를 내가 통제하고 싶을 때 (RL, GAN 등) | 일반적인 지도학습 |

수식이 어떻게 코드로 흘러가는지 이해하려면 저수준이 필수, 실무 코드를 빠르게 짜려면 Keras가 표준.

## 5. 강의의 TF 1.x 코드 → 2.x 매핑

원본:
```python
# TF 1.x — 실행 안 됨
x = tf.placeholder(tf.float32)
y = tf.placeholder(tf.float32)
W = tf.Variable(tf.random_normal([1]))
b = tf.Variable(tf.random_normal([1]))
h = W * x + b
cost = tf.reduce_mean(tf.square(h - y))
optimizer = tf.train.GradientDescentOptimizer(0.1)
train = optimizer.minimize(cost)

with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    for step in range(2001):
        sess.run(train, feed_dict={x: [1,2,3], y: [1,2,3]})
```

2.x 동치:
- `placeholder` + `feed_dict` → 그냥 텐서/배열을 함수에 전달
- `train.GradientDescentOptimizer` → `tf.keras.optimizers.SGD`
- `Session` / `global_variables_initializer` → 불필요
- `optimizer.minimize` → `GradientTape` + `assign_sub` 또는 `optimizer.apply_gradients` 또는 `model.fit`

## 마무리 — 체크리스트

- [ ] 가설 $H(x)=Wx+b$ 와 MSE cost 식을 보고 코드로 옮길 수 있다
- [ ] `tf.GradientTape` 으로 W, b 의 그래디언트를 받아 봤다
- [ ] cost 곡선이 단조 감소하는지 그림으로 확인했다
- [ ] 같은 문제를 `tf.keras.Sequential([Dense(1)])` 로도 풀어 봤다
- [ ] learning rate 를 바꿔서 발산/수렴 차이를 직접 봤다 (예: 1.5)